In [ ]:
import os
import time
import nest_asyncio
from ollama import AsyncClient
import pandas as pd
import asyncio
import subprocess


nest_asyncio.apply()
MODEL_NAME = "qwen2.5-coder:1.5b"
os.system("pkill ollama")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(15)
os.system(f"ollama pull {MODEL_NAME}")
BATCH_SIZE = 5
async def classify_row(client, idx, instruction, code):
    prompt = f"""
Classify into ONE Broad Category: [array,graph,tree,sorting,searching,hashmap,dynamic programming,string,Web Dev,Machine Learning]
Task: {str(instruction)[:400]}
Code: {str(code)[:300]}
Return ONLY:
Category
"""
    try:
        response = await client.generate(model=MODEL_NAME, prompt=prompt, options={"temperature": 0})
        text = response['response'].strip().split("\n")[0]
        # Live feedback
        print(f"\nRow {idx+1}: Predicted -> {text}")
        return idx, text
    except Exception as e:
        print(f"\nRow {idx+1}: ERROR -> {e}")
        return idx, f"Error | {str(e)[:50]}"

async def run_pipeline(df):
    client = AsyncClient()
    tasks = [classify_row(client, idx, row['instruction'], row['output']) for idx, row in df.iterrows()]

    results = []
    for i in range(0, len(tasks), BATCH_SIZE):
        batch = tasks[i:i+BATCH_SIZE]
        batch_results = await asyncio.gather(*batch)
        results.extend(batch_results)
        print(f"Processed {min(i+BATCH_SIZE, len(df))}/{len(df)} rows so far")
    return results


print("Starting classification...\n")
final_results = await run_pipeline(df)
results_dict = dict(final_results)
df["classification_raw"] = df.index.map(results_dict)
df[["category", "topic"]] = df["classification_raw"].str.split("|", n=1, expand=True)
df["category"] = df["category"].str.strip()
df["topic"] = df["topic"].str.strip()
print("\nClassification Complete!")
print("\nCategory counts:")
print(df["category"].value_counts())
print("\n--- Sample Predictions ---")
for i in range(min(5, len(df))):
    print(f"\nInstruction: {df.loc[i, 'instruction']}")
    print(f"Code: {df.loc[i, 'output'][:200]}...")
    print(f"Predicted: {df.loc[i, 'classification_raw']}")